In [4]:
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.datasets import RetrievalDataset, MassSpecDataset
from chemembed_transforms import ChemEmbedSpecTransform, ChemEmbedMolTransform
from pathlib import Path
from ann_model_massspecgym import AnnRetrieval
from pytorch_lightning import Trainer
from massspecgym.models.base import Stage
import pandas as pd

In [ ]:
# Train dataset, without reference candidates
train_dataset = MassSpecDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=ChemEmbedMolTransform(model_path='Mol2Vec_model/model_300dim.pkl'),
)
data_module_train = MassSpecDataModule(dataset=train_dataset, batch_size=32)

data_module_train.prepare_data()  # Explicit call needed for validate before fit
data_module_train.setup()  # Explicit call needed for validate before fit

# Init model
model = AnnRetrieval(
    log_only_loss_at_stages=[Stage.TRAIN, Stage.VAL]
)

# Init trainer
trainer = Trainer(
    accelerator="auto", # Automatically use GPU if available
    max_epochs=15,        
    log_every_n_steps=1
)

# Validation before training
trainer.validate(model, datamodule=data_module_train)

# Train
trainer.fit(model, datamodule=data_module_train)


# Save model checkpoint
trainer.save_checkpoint("ANN_trained_on_massspecgym.ckpt")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 608/608 [05:00<00:00,  2.03it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss            111.49406433105469
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


Restoring states from the checkpoint path at lightning_logs/version_5/checkpoints/epoch=2-step=18201.ckpt
c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:361: The dirpath has changed from 'c:\\Users\\aa\\Desktop\\TFM\\Final_MassSpecGym\\lightning_logs\\version_5\\checkpoints' to 'c:\\Users\\aa\\Desktop\\TFM\\Final_MassSpecGym\\lightning_logs\\version_7\\checkpoints', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won't be reloaded. Only `best_model_path` will be reloaded.

  | Name | Type      | Params
-----------------------------------
0 | ann  | ANN_Class | 145 M 
-----------------------------------
145 M     Trainable params
0         Non-trainable params
145 M     Total params
583.071   Total estimated model params size (MB)
Restored all states from the checkpoint at lightning_logs/version_5/checkpoints/epoch=2-step=18201.ckpt


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 3: 100%|██████████| 6067/6067 [6:25:07<00:00,  0.26it/s, v_num=7, train_loss=3.830]  
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:   6%|▌         | 346/6067 [11:09:20<184:27:19,  0.01it/s, v_num=7, train_loss=2.770, val_loss=32.20]

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [ ]:
# Standard test 
test_dataset = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=ChemEmbedMolTransform(model_path='Mol2Vec_model/model_300dim.pkl'),
)
data_module_test = MassSpecDataModule(dataset=test_dataset, batch_size=32)
data_module_test.setup(stage="test")

test_results = trainer.test(model, datamodule=data_module_test)
results_df = pd.DataFrame(test_results)
results_df.to_csv("masspecgym_ann_results.csv", index=False)

In [ ]:
# BONUS test with candidates (same molecular formula)
test_dataset_bonus = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=ChemEmbedMolTransform(model_path='Mol2Vec_model/model_300dim.pkl'),
    candidates_pth='bonus'
)
data_module_test_bonus = MassSpecDataModule(dataset=test_dataset_bonus, batch_size=32)
data_module_test_bonus.setup(stage="test")

test_results_bonus = trainer.test(model, datamodule=data_module_test_bonus)
results_df_bonus = pd.DataFrame(test_results_bonus)
results_df_bonus.to_csv("masspecgym_ann_results_bonus.csv", index=False)